# Phân tích Spotify bằng Apache Spark
Chạy cell thiết lập repository của notebook 01 trong runtime này trước (không cần GPU).
Notebook này chạy Spark local trên Colab. Đây không phải benchmark cluster nhiều máy.


In [ ]:
import subprocess, sys, os
from pathlib import Path
ROOT = Path('/content/music-insight')
assert (ROOT/'pyproject.toml').exists(), 'Chạy cell thiết lập repository trong notebook 01 trước'
os.chdir(ROOT)
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','openjdk-17-jdk-headless'], check=True)
os.environ['JAVA_HOME']='/usr/lib/jvm/java-17-openjdk-amd64'
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[spark]'], check=True)
from music.data import download
download('data/raw/dataset.csv')


In [ ]:
from pipelines.batch import run
run('data/raw/dataset.csv', 'data/lake')


In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master('local[*]').appName('spotify-eda').getOrCreate()
tracks = spark.read.parquet('data/lake/silver/tracks')
genres = spark.read.parquet('data/lake/gold/genre_stats')
tracks.printSchema()
tracks.select('energy','valence','tempo','popularity').summary().show()
print('Energy/loudness correlation:', tracks.stat.corr('energy','loudness'))
display(genres.orderBy(F.desc('avg_popularity')).limit(20).toPandas())


In [ ]:
import matplotlib.pyplot as plt
chart = genres.orderBy(F.desc('avg_popularity')).limit(20).toPandas().sort_values('avg_popularity')
chart.plot.barh(x='track_genre', y='avg_popularity', figsize=(9,7), legend=False)
plt.xlabel('Popularity trung bình tại snapshot'); plt.tight_layout()
Path('reports/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('reports/figures/genre-popularity.png', dpi=150)
plt.show()


In [ ]:
# Ghi nhận benchmark thực đo: cùng input và cùng phép tính.
import time, json
results=[]
for partitions in [2,4,8]:
    start=time.perf_counter()
    count=tracks.repartition(partitions).groupBy('artists').agg(F.avg('energy')).count()
    results.append({'partitions':partitions,'seconds':time.perf_counter()-start,'artist_groups':count})
print(results)
Path('reports/benchmark-local.json').write_text(json.dumps(results,indent=2))
spark.stop()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, datetime
target=Path('/content/drive/MyDrive/MusicInsight/spark')/datetime.datetime.now().strftime('%Y%m%dT%H%M%S')
target.mkdir(parents=True)
shutil.copytree('data/lake/gold',target/'gold')
shutil.copytree('reports/figures',target/'figures')
shutil.copy2('reports/benchmark-local.json',target/'benchmark-local.json')
print(target)
